In [1]:
import os

In [2]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_TOKEN")
os.environ["GROQ_API_KEY"] = user_secrets.get_secret("groq key")

In [3]:
!git clone https://github.com/tweylib/EAT_BART.git
%cd EAT_BART
# !pip install -r requirements.txt

%cd /kaggle/working/EAT_BART
!git checkout ablation/encoder-eat
!git pull
!pip install -r requirements.txt

Cloning into 'EAT_BART'...
remote: Enumerating objects: 337, done.
remote: Counting objects: 100% (337/337), done.
remote: Compressing objects: 100% (158/158), done.
remote: Total 337 (delta 212), reused 267 (delta 149), pack-reused 0 (from 0)
Receiving objects: 100% (337/337), 66.07 KiB | 8.26 MiB/s, done.
Resolving deltas: 100% (212/212), done.
/kaggle/working/EAT_BART
/kaggle/working/EAT_BART
Branch 'ablation/encoder-eat' set up to track remote branch 'ablation/encoder-eat' from 'origin'.
Switched to a new branch 'ablation/encoder-eat'
Already up to date.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 7.8 MB/s eta 0:00:00


### : ) Train

In [4]:
!python scripts/train.py --config configs/kaggle_encoder_only_5epoch.yaml

config.json: 1.72kB [00:00, 879kB/s]
vocab.json: 899kB [00:00, 41.9MB/s]
merges.txt: 456kB [00:00, 15.9MB/s]
tokenizer.json: 1.36MB [00:00, 11.2MB/s]
model.safetensors: 100%|██████████████████████| 558M/558M [00:04<00:00, 138MB/s]
Loading weights: 100%|█| 259/259 [00:00<00:00, 1659.13it/s, Materializing param=
  0%|                                                  | 0/2485 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
{'loss': '58.15', 'grad_norm': '43.46', 'learning_rate': '2.941e-05', 'epoch': '0.1006'}
{'loss': '50.48', 'grad_norm': '46.31', 'learning_rate': '2.88e-05', 'epoch': '0.2012'}
{'loss': '49.06', 'grad_norm': '50.73', 'learning_rate': '2.82e-05', 'epoch': '0.3018'}
{'loss': '47.32', 'grad_norm': '39.2', 'learning_rate': '2.76e-05', 'epoch': 

### ; ) Evaluate

In [5]:
!python scripts/evaluate.py --config configs/kaggle_encoder_only_5epoch_experiment_evaluate.yaml
!python scripts/score_generations.py --config configs/kaggle_encoder_only_5epoch_experiment_score.yaml

config.json: 100%|█████████████████████████████| 482/482 [00:00<00:00, 2.15MB/s]
tokenizer_config.json: 100%|█████████████████| 25.0/25.0 [00:00<00:00, 79.6kB/s]
vocab.json: 899kB [00:00, 28.2MB/s]
merges.txt: 456kB [00:00, 6.96MB/s]
tokenizer.json: 1.36MB [00:00, 16.8MB/s]
model.safetensors: 100%|████████████████████| 1.42G/1.42G [00:05<00:00, 272MB/s]
Loading weights: 100%|█| 389/389 [00:00<00:00, 1453.37it/s, Materializing param=
RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored w

### : > LLMs Judges

In [6]:
!python scripts/judge_generations.py --config configs/kaggle_encoder_only_5epoch_experiment_judge_groq.yaml
!python scripts/judge_generations.py --config configs/kaggle_encoder_only_5epoch_experiment_judge_groq_gpt_oss.yaml

!python scripts/aggregate_judges.py --config configs/kaggle_encoder_only_5epoch_experiment_judge_groq_2judge_aggregate.yaml

num_requested_examples: 100.0000
num_judged_examples: 100.0000
num_failed_examples: 0.0000
llm_empathy: 3.7800
llm_coherence: 4.0400
llm_safety: 4.7200
num_requested_examples: 100.0000
num_judged_examples: 12.0000
num_failed_examples: 88.0000
llm_empathy: 3.0000
llm_coherence: 3.7500
llm_safety: 4.8333
num_judges: 2.0000
num_completed_judges: 2.0000
judges: llama-3.3-70b-versatile;openai/gpt-oss-120b
completed_judges: llama-3.3-70b-versatile;openai/gpt-oss-120b
llm_empathy_mean_across_judges: 3.3900
llm_coherence_mean_across_judges: 3.8950
llm_safety_mean_across_judges: 4.7767


In [7]:
from pathlib import Path

for path in [
    "/kaggle/working/models/eat_bart_encoder_only_5epoch",
    "/kaggle/working/reports/eat_bart_encoder_only_5epoch_experiment_generations.csv",
    "/kaggle/working/reports/eat_bart_encoder_only_5epoch_experiment_metrics.csv",
    "/kaggle/working/reports/eat_bart_encoder_only_5epoch_experiment_llm_judge_groq.csv",
    "/kaggle/working/reports/eat_bart_encoder_only_5epoch_experiment_llm_judge_groq_gpt_oss.csv",
    "/kaggle/working/reports/eat_bart_encoder_only_5epoch_experiment_llm_judge_groq_2judge_aggregate.csv",
]:
    p = Path(path)
    print(path, "EXISTS" if p.exists() else "MISSING")

/kaggle/working/models/eat_bart_encoder_only_5epoch EXISTS
/kaggle/working/reports/eat_bart_encoder_only_5epoch_experiment_generations.csv EXISTS
/kaggle/working/reports/eat_bart_encoder_only_5epoch_experiment_metrics.csv EXISTS
/kaggle/working/reports/eat_bart_encoder_only_5epoch_experiment_llm_judge_groq.csv EXISTS
/kaggle/working/reports/eat_bart_encoder_only_5epoch_experiment_llm_judge_groq_gpt_oss.csv EXISTS
/kaggle/working/reports/eat_bart_encoder_only_5epoch_experiment_llm_judge_groq_2judge_aggregate.csv EXISTS


In [8]:
!cd /kaggle/working && zip -r eat_bart_encoder_only_5epoch_experiment_outputs.zip models/eat_bart_encoder_only_5epoch reports

  adding: models/eat_bart_encoder_only_5epoch/ (stored 0%)
  adding: models/eat_bart_encoder_only_5epoch/tokenizer_config.json (deflated 49%)
  adding: models/eat_bart_encoder_only_5epoch/training_args.bin (deflated 53%)
  adding: models/eat_bart_encoder_only_5epoch/checkpoint-1988/ (stored 0%)
  adding: models/eat_bart_encoder_only_5epoch/checkpoint-1988/tokenizer_config.json (deflated 49%)
  adding: models/eat_bart_encoder_only_5epoch/checkpoint-1988/training_args.bin (deflated 53%)
  adding: models/eat_bart_encoder_only_5epoch/checkpoint-1988/trainer_state.json (deflated 75%)
  adding: models/eat_bart_encoder_only_5epoch/checkpoint-1988/rng_state.pth (deflated 26%)
  adding: models/eat_bart_encoder_only_5epoch/checkpoint-1988/generation_config.json (deflated 46%)
  adding: models/eat_bart_encoder_only_5epoch/checkpoint-1988/tokenizer.json (deflated 82%)
  adding: models/eat_bart_encoder_only_5epoch/checkpoint-1988/scheduler.pt (deflated 61%)
  adding: models/eat_bart_encoder_only_5e

-----------------------------------------------------------

## before code

In [9]:
# !python scripts/train.py --config configs/kaggle_encoder_only_5epoch.yaml

In [10]:
# !python scripts/evaluate.py --config configs/kaggle_encoder_only_5epoch_evaluate.yaml
# !python scripts/score_generations.py --config configs/kaggle_encoder_only_5epoch_score.yaml